In [2]:
:dep md5 = "=0.7.0"
:dep regex = "=1.9.6"
:dep anyhow = "=1.0.75"

use anyhow::{anyhow, Result};
use regex::Regex;
use std::fs;
use std::io::{Read, Write};
use std::net::TcpStream;
use std::time::Duration;

In [3]:
// --- PCAP パーサー ---
fn parse_pcap_payloads(data: &[u8]) -> Vec<Vec<u8>> {
    let mut payloads = Vec::new();
    if data.len() < 24 { return payloads; }
    let magic = u32::from_le_bytes([data[0], data[1], data[2], data[3]]);
    let le = magic == 0xa1b2c3d4 || magic == 0xa1b23c4d;
    let read_u32 = |buf: &[u8], off: usize| -> u32 {
        if le { u32::from_le_bytes([buf[off], buf[off+1], buf[off+2], buf[off+3]]) }
        else  { u32::from_be_bytes([buf[off], buf[off+1], buf[off+2], buf[off+3]]) }
    };
    let mut pos = 24usize;
    while pos + 16 <= data.len() {
        let incl_len = read_u32(data, pos + 8) as usize;
        let record_end = pos + 16 + incl_len;
        if record_end > data.len() { break; }
        payloads.push(data[pos + 16..record_end].to_vec());
        pos = record_end;
    }
    payloads
}

// --- HA1 抽出 ---
fn extract_ha1_from_pcap(pcap_path: &str) -> Result<String> {
    let data = fs::read(pcap_path)
        .map_err(|e| anyhow!("Cannot read '{}': {}", pcap_path, e))?;
    let ha1_re = Regex::new(r"q9:secret:([a-f0-9]{32})")?;
    for payload in parse_pcap_payloads(&data) {
        let text = String::from_utf8_lossy(&payload);
        if let Some(cap) = ha1_re.captures(&text) {
            return Ok(cap[1].to_string());
        }
    }
    Err(anyhow!("HA1 not found in pcap."))
}

// --- HTTP (TcpStream) ---
fn http_get(host: &str, port: u16, path: &str, extra_headers: &[(&str, &str)]) -> Result<String> {
    let mut stream = TcpStream::connect(format!("{}:{}", host, port))?;
    stream.set_read_timeout(Some(Duration::from_secs(15)))?;
    stream.set_write_timeout(Some(Duration::from_secs(10)))?;
    let mut req = format!("GET {} HTTP/1.1\r\nHost: {}\r\nConnection: close\r\n", path, host);
    for (k, v) in extra_headers { req.push_str(&format!("{}: {}\r\n", k, v)); }
    req.push_str("\r\n");
    stream.write_all(req.as_bytes())?;
    let mut buf = Vec::new();
    stream.read_to_end(&mut buf)?;
    Ok(String::from_utf8_lossy(&buf).to_string())
}

fn parse_status(r: &str) -> u16 {
    r.lines().next().and_then(|l| l.split_whitespace().nth(1)).and_then(|s| s.parse().ok()).unwrap_or(0)
}
fn get_header<'a>(r: &'a str, name: &str) -> Option<&'a str> {
    let lower = name.to_lowercase();
    r.lines().find_map(|line| {
        line.find(':').and_then(|c| {
            (line[..c].trim().to_lowercase() == lower).then(|| line[c+1..].trim())
        })
    })
}
fn get_body(r: &str) -> &str {
    r.find("\r\n\r\n").map(|p| &r[p+4..]).unwrap_or("")
}

// --- Digest 計算 ---
fn extract_param(header: &str, name: &str) -> Result<String> {
    let re = Regex::new(&format!(r#"{}="?([^",\s]+)"?"#, regex::escape(name)))?;
    re.captures(header).and_then(|c| c.get(1)).map(|m| m.as_str().to_string())
        .ok_or_else(|| anyhow!("'{}' not found", name))
}
fn compute_digest_response(ha1: &str, nonce: &str, nc: &str, cnonce: &str, qop: &str, method: &str, uri: &str) -> String {
    let ha2  = format!("{:x}", md5::compute(format!("{}:{}", method, uri)));
    format!("{:x}", md5::compute(format!("{}:{}:{}:{}:{}:{}", ha1, nonce, nc, cnonce, qop, ha2)))
}
fn build_authorization(username: &str, realm: &str, nonce: &str, uri: &str,
                        algorithm: &str, response: &str, qop: &str, nc: &str, cnonce: &str) -> String {
    format!(
        r#"Digest username="{}", realm="{}", nonce="{}", uri="{}", algorithm={}, response="{}", qop={}, nc={}, cnonce="{}""#,
        username, realm, nonce, uri, algorithm, response, qop, nc, cnonce
    )
}

In [5]:
let pcap_path = "q9.pcap";

let ha1 = extract_ha1_from_pcap(pcap_path)?;
println!("[+] HA1: {}", ha1);

let host = "ctfq.u1tramarine.blue";
let path = "/q9/flag.html";

let resp1 = http_get(host, 80, path, &[])?;
let www_auth = get_header(&resp1, "WWW-Authenticate").unwrap().to_string();
println!("[+] WWW-Authenticate: {}", www_auth);

let realm     = extract_param(&www_auth, "realm")?;
let nonce     = extract_param(&www_auth, "nonce")?;
let qop       = extract_param(&www_auth, "qop")?;
let algorithm = extract_param(&www_auth, "algorithm")?;

let digest_resp = compute_digest_response(&ha1, &nonce, "00000001", "abcdef0123456789", &qop, "GET", path);
let auth = build_authorization("q9", &realm, &nonce, path, &algorithm, &digest_resp, &qop, "00000001", "abcdef0123456789");

let resp2 = http_get(host, 80, path, &[("Authorization", &auth)])?;
println!("HTTP {}", parse_status(&resp2));
fs::write("flag.txt", get_body(&resp2)).unwrap();

[+] HA1: c627e19450db746b739f41b64097d449
[+] WWW-Authenticate: Digest realm="secret", nonce="BuUx8oxQBgA=d117e4c40e587b2304f6545590c82eca0744808e", algorithm=MD5, qop="auth"
HTTP 200
